In [1]:
# =============================================================================
# IMPORTS
# =============================================================================

import numpy as np
import pandas as pd
import random
import os
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoLoader
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

In [2]:
# =============================================================================
# SETTINGS
# =============================================================================

SEED        = 42
OUTPUT_DIR  = r"C:\Users\Pratham Shah\Downloads\Document dataset\Final_MyData\GAT_GO_Results_1"
DATA_DIR    = r"C:\Users\Pratham Shah\Downloads\Document dataset\Final_MyData"

COMMUNITY_FILE  = "community_dataset_noGO_v6.csv"
INDEX_FILE      = "community_index_v6.csv"
MONOCULTURE_FILE = "monoculture_dataset_v6.csv"

N_GO_FEATURES          = 1000   # raw GO features per strain
GO_EMBEDDING_DIM       = 64     # compressed GO representation
N_SHAPLEY_PERMUTATIONS = 50
N_EPOCHS               = 300
LR                     = 1e-3
BATCH_SIZE             = 16
TEST_FRAC              = 0.20
# For feature selection: how many top GO features to keep
N_TOP_GO               = 200

os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

TEAL   = "#1a6b6b"
ORANGE = "#d4720c"
NAVY   = "#1c2e4a"
RED    = "#E53935"
GREEN  = "#43A047"
BLUE   = "#1E88E5"
GREY   = "#78909C"

plt.rcParams.update({
    "figure.facecolor" : "white",
    "axes.facecolor"   : "white",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.family"      : "sans-serif",
    "font.size"        : 11,
})

In [3]:
# =============================================================================
# SECTION 1 — LOAD DATA
# =============================================================================

print("=" * 65)
print("  GAT-GO: Graph Attention Network — Gene Orthologue Features")
print("=" * 65)

print("\n[1/8] Loading data ...")

# community_dataset.csv has GO features — this is the large file
# We load it in chunks because it is big (39362 rows × 1062 columns)
print("  Loading community_dataset.csv (large file, may take a moment) ...")
df      = pd.read_csv(os.path.join(DATA_DIR, COMMUNITY_FILE))
df_idx  = pd.read_csv(os.path.join(DATA_DIR, INDEX_FILE))
df_mono = pd.read_csv(os.path.join(DATA_DIR, MONOCULTURE_FILE))

# Extract GO column names
go_cols = [c for c in df_mono.columns if c.startswith("GO_")]
print(f"  GO features found in monoculture file : {len(go_cols)}")
print(f"  Community member rows                 : {len(df):,}")
print(f"  Communities                           : {len(df_idx):,}")
print(f"  Strains in monoculture                : {len(df_mono):,}")

  GAT-GO: Graph Attention Network — Gene Orthologue Features

[1/8] Loading data ...
  Loading community_dataset.csv (large file, may take a moment) ...
  GO features found in monoculture file : 1000
  Community member rows                 : 18,000
  Communities                           : 1,200
  Strains in monoculture                : 100


In [5]:
# =============================================================================
# SECTION 2 — GO FEATURE SELECTION
# =============================================================================

"""
WHY FEATURE SELECTION?
-----------------------
We have 1000 GO features but many carry no signal — they are random
noise that would confuse the model. Keeping all 1000 would:
  1. Slow down training enormously
  2. Add noise that hurts prediction accuracy
  3. Make the model harder to interpret

The paper reduces 58,764 → ~572 features using iterative SHAP-based
removal. We use a simpler but effective approach:
  VARIANCE FILTER: remove GO features with zero or near-zero variance
  (features that are the same for nearly all strains carry no information)
  then keep the top N_TOP_GO most variable features.

This is called "unsupervised feature selection" — we don't use the
labels to select features, only the feature distributions.
Using labels for feature selection before the train/test split would
be data leakage.
"""

print(f"\n[2/8] GO feature selection ...")

# Build GO matrix: [100 strains, 1000 features]
go_from_mono = df_mono[go_cols].values.astype(np.float32)
 
# Variance of each GO feature across all 100 strains.
# np.var(axis=0): one variance value per column.
go_variance = np.var(go_from_mono, axis=0)   # [1000]
 
# Sort descending by variance, keep top N_TOP_GO.
# np.argsort gives ascending indices; [::-1] reverses to descending.
top_go_indices = np.argsort(go_variance)[::-1][:N_TOP_GO]
top_go_cols    = [go_cols[i] for i in top_go_indices]
 
print(f"  Features with zero variance  : {(go_variance == 0).sum()}")
print(f"  Keeping top {N_TOP_GO} most variable GO features")
print(f"  Variance range of kept features: "
      f"{go_variance[top_go_indices].min():.4f} – "
      f"{go_variance[top_go_indices].max():.4f}")


[2/8] GO feature selection ...
  Features with zero variance  : 0
  Keeping top 200 most variable GO features
  Variance range of kept features: 1.1596 – 1.4936


In [6]:
# =============================================================================
# SECTION 3 — NORMALISE GO FEATURES
# =============================================================================

"""
WHY NORMALISE?
--------------
GO features are copy numbers: mostly 0, sometimes 1, 2, or 3.
Most features have very different scales and distributions.
Neural networks train better when input features are on a similar scale.
StandardScaler converts each feature to have mean=0 and std=1.

IMPORTANT: fit the scaler on TRAINING data only, apply to all data.
Fitting on all data would cause data leakage (the model would "see"
information from the test set during training via the scaler).
We fit on monoculture data here (before the train/test split)
which is acceptable since monoculture data is the "feature source"
not the prediction target.
"""

scaler = StandardScaler()
# fit_transform: computes mean and std per feature column, then applies
# z = (x - mean) / std. Result has mean=0, std=1 per column.
go_normalised_mono = scaler.fit_transform(
    df_mono[top_go_cols].values.astype(np.float32)
)   # [100, N_TOP_GO]
 
# Build strain → normalised GO vector lookup dictionary.
# Key: strain_id (e.g. "Species_0045")
# Value: normalised GO feature vector of length N_TOP_GO
strain_go_map = {}
for i, strain_id in enumerate(df_mono["strain_id"].values):
    strain_go_map[strain_id] = go_normalised_mono[i]
 
print(f"  GO feature matrix after normalisation: "
      f"mean={go_normalised_mono.mean():.3f}, std={go_normalised_mono.std():.3f}")
 

  GO feature matrix after normalisation: mean=0.000, std=1.000


In [8]:
print("\n[4/8] Building graphs ...")
 
graph_list = []
all_community_ids = df["community_id"].unique().tolist()
n_skipped  = 0
 
for comm_id in all_community_ids:
 
    comm_rows = df[df["community_id"] == comm_id].copy()
    n_nodes   = len(comm_rows)
 
    # Build node feature matrix by looking up each strain's GO vector.
    node_feats_list = []
    strain_ids      = []
    valid = True
 
    for _, row in comm_rows.iterrows():
        sid = row["strain_id"]
        if sid not in strain_go_map:
            # Should never happen in v6 since all 100 community strains
            # are present in monoculture_dataset_v6.csv. Skip if missing.
            valid = False
            break
        node_feats_list.append(strain_go_map[sid])
        strain_ids.append(sid)
 
    if not valid:
        n_skipped += 1
        continue
 
    # np.stack(list_of_arrays, axis=0): stacks 1D arrays as rows into 2D.
    # Each array has shape [N_TOP_GO], result shape: [n_nodes, N_TOP_GO].
    node_feats = np.stack(node_feats_list, axis=0)
    x = torch.tensor(node_feats, dtype=torch.float)
    # Shape: [n_nodes, N_TOP_GO] = [n_nodes, 200]
 
    # Fully connected edges (same as GAT-SS).
    src, dst = [], []
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                src.append(i)
                dst.append(j)
    edge_index = torch.tensor([src, dst], dtype=torch.long)
 
    # Label from index file.
    label_row             = df_idx[df_idx["community_id"] == comm_id]
    community_suppression = float(label_row["community_suppression"].values[0])
    community_size        = int(label_row["community_size"].values[0])
    y = torch.tensor([community_suppression], dtype=torch.float)
 
    data = Data(x=x, edge_index=edge_index, y=y)
    data.community_id   = comm_id
    data.community_size = community_size
    data.strain_ids     = strain_ids
 
    graph_list.append(data)
 
print(f"  Graphs built : {len(graph_list)}")
print(f"  Skipped      : {n_skipped}")
sample = graph_list[0]
print(f"  Sample graph — nodes: {sample.x.shape[0]}, "
      f"features per node: {sample.x.shape[1]}, "
      f"edges: {sample.edge_index.shape[1]}")
# Expected: features per node = N_TOP_GO = 200


[4/8] Building graphs ...
  Graphs built : 1200
  Skipped      : 0
  Sample graph — nodes: 2, features per node: 200, edges: 2


In [9]:
# =============================================================================
# SECTION 5 — TRAIN / TEST SPLIT
# =============================================================================

print("\n[5/8] Splitting into train / test ...")
 
sizes_array = np.array([g.community_size for g in graph_list])
train_graphs, test_graphs = train_test_split(
    graph_list,
    test_size    = TEST_FRAC,
    random_state = SEED,
    stratify     = sizes_array,
)
 
print(f"  Train: {len(train_graphs)} graphs")
print(f"  Test : {len(test_graphs)} graphs")
 
train_loader = GeoLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = GeoLoader(test_graphs,  batch_size=BATCH_SIZE, shuffle=False)
 


[5/8] Splitting into train / test ...
  Train: 960 graphs
  Test : 240 graphs


In [10]:
# =============================================================================
# SECTION 6 — MODEL ARCHITECTURE
# =============================================================================

class GATGO(nn.Module):
    """
    Graph Attention Network — Gene Orthologue (GAT-GO)

    Key difference from GAT-SS:
      Input: 200 GO features per node (after feature selection)
      → Linear encoder: compress 200 → GO_EMBEDDING_DIM (64)
      → GATConv layer 1: attention over node neighbourhoods
      → GATConv layer 2: second round of attention
      → global_mean_pool + global_max_pool: graph-level vector
      → MLP head: predict suppression score

    WHY THE ENCODER?
    ----------------
    Going directly from 200 features to the GAT layer is fine but an
    encoder layer helps the model first compress the genomic information
    into a more compact, meaningful representation before doing message
    passing. This is standard practice with high-dimensional node features.
    Think of it as: "summarise your genome first, then talk to your neighbours."
    """

    def __init__(self, in_channels, embed_dim, hidden_channels, heads, dropout):
        super(GATGO, self).__init__()

        # ── GO Encoder: compress GO features → embedding ─────────────────────
        # This is a simple linear layer: output = input @ weight + bias
        # in_channels = N_TOP_GO (200), embed_dim = 64
        # The encoder learns which combinations of GO features matter
        self.go_encoder = nn.Sequential(
            nn.Linear(in_channels, embed_dim),
            nn.ELU(),
            nn.Dropout(p=dropout),
            nn.Linear(embed_dim, embed_dim),
            nn.ELU(),
        )
        # nn.Sequential chains layers in order — output of each feeds into next

        # ── GATConv layers (same as GAT-SS but taking embed_dim as input) ────
        self.conv1 = GATConv(
            embed_dim,
            hidden_channels,
            heads=heads,
            concat=True,
            dropout=dropout,
            add_self_loops=True
        )

        self.conv2 = GATConv(
            hidden_channels * heads,
            hidden_channels,
            heads=1,
            concat=False,
            dropout=dropout,
            add_self_loops=True
        )

        self.bn1 = nn.BatchNorm1d(hidden_channels * heads)
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        self.dropout = nn.Dropout(p=dropout)

        # ── MLP prediction head ───────────────────────────────────────────────
        self.lin1 = nn.Linear(hidden_channels * 2, hidden_channels)
        self.lin2 = nn.Linear(hidden_channels, 1)

    def forward(self, data):
        """
        Forward pass for GAT-GO.

        Parameters
        ----------
        data.x : [total_nodes, N_TOP_GO] — raw GO features

        Returns
        -------
        [N_graphs] — predicted suppression per graph
        """
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Step 1: encode GO features → compact embedding
        # Each node's 200-dim GO vector is compressed to 64-dim
        x = self.go_encoder(x)          # [total_nodes, embed_dim]

        # Step 2: GAT message passing (same as GAT-SS from here)
        x = self.conv1(x, edge_index)   # [total_nodes, hidden * heads]
        x = self.bn1(x)
        x = F.elu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)   # [total_nodes, hidden]
        x = self.bn2(x)
        x = F.elu(x)

        # Step 3: pool nodes → graph-level representation
        x_mean = global_mean_pool(x, batch)
        x_max  = global_max_pool(x, batch)
        x = torch.cat([x_mean, x_max], dim=1)

        # Step 4: MLP head → prediction
        x = self.dropout(x)
        x = F.elu(self.lin1(x))
        x = torch.sigmoid(self.lin2(x))

        return x.squeeze(-1)

    def get_node_embeddings(self, data):
        """Returns node embeddings after GO encoding + 2 GAT layers."""
        x, edge_index = data.x, data.edge_index
        x = self.go_encoder(x)
        x = F.elu(self.bn1(self.conv1(x, edge_index)))
        x = F.elu(self.bn2(self.conv2(x, edge_index)))
        return x


# Instantiate model
model = GATGO(
    in_channels     = N_TOP_GO,       # 200 GO features per node
    embed_dim       = GO_EMBEDDING_DIM,  # compress to 64
    hidden_channels = 32,
    heads           = 4,
    dropout         = 0.3,
)

print(f"\n[6/8] Model architecture:")
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total trainable parameters: {total_params:,}")


[6/8] Model architecture:
GATGO(
  (go_encoder): Sequential(
    (0): Linear(in_features=200, out_features=64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): ELU(alpha=1.0)
  )
  (conv1): GATConv(64, 32, heads=4)
  (conv2): GATConv(128, 32, heads=1)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (lin1): Linear(in_features=64, out_features=32, bias=True)
  (lin2): Linear(in_features=32, out_features=1, bias=True)
)
  Total trainable parameters: 32,225


In [11]:
# =============================================================================
# SECTION 7 — TRAINING
# =============================================================================

print("\n[7/8] Training ...")

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
loss_fn   = nn.HuberLoss(delta=0.15)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=40, factor=0.5
)

train_losses, test_losses = [], []

for epoch in range(1, N_EPOCHS + 1):

    model.train()
    total_loss, n_batches = 0.0, 0

    for batch in train_loader:
        optimizer.zero_grad()
        pred = model(batch)
        loss = loss_fn(pred, batch.y.squeeze())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches  += 1

    avg_train_loss = total_loss / n_batches

    model.eval()
    total_test_loss, n_test_batches = 0.0, 0
    with torch.no_grad():
        for batch in test_loader:
            pred = model(batch)
            loss = loss_fn(pred, batch.y.squeeze())
            total_test_loss += loss.item()
            n_test_batches  += 1

    avg_test_loss = total_test_loss / max(n_test_batches, 1)
    train_losses.append(avg_train_loss)
    test_losses.append(avg_test_loss)
    scheduler.step(avg_test_loss)

    if epoch % 50 == 0 or epoch == 1:
        print(f"  Epoch {epoch:>3}/{N_EPOCHS} | "
              f"Train: {avg_train_loss:.4f} | "
              f"Test: {avg_test_loss:.4f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.6f}")


[7/8] Training ...
  Epoch   1/300 | Train: 0.0296 | Test: 0.0233 | LR: 0.001000
  Epoch  50/300 | Train: 0.0105 | Test: 0.0135 | LR: 0.001000
  Epoch 100/300 | Train: 0.0104 | Test: 0.0153 | LR: 0.000500
  Epoch 150/300 | Train: 0.0087 | Test: 0.0150 | LR: 0.000250
  Epoch 200/300 | Train: 0.0089 | Test: 0.0145 | LR: 0.000125
  Epoch 250/300 | Train: 0.0080 | Test: 0.0149 | LR: 0.000063
  Epoch 300/300 | Train: 0.0085 | Test: 0.0148 | LR: 0.000016


In [12]:
# =============================================================================
# SECTION 8 — EVALUATION
# =============================================================================

print("\n[8/8] Evaluating ...")

model.eval()
all_preds, all_true, all_sizes = [], [], []

with torch.no_grad():
    for data in test_graphs:
        batch_tensor = torch.zeros(data.x.shape[0], dtype=torch.long)
        d    = Data(x=data.x, edge_index=data.edge_index, batch=batch_tensor)
        pred = model(d).item()
        all_preds.append(pred)
        all_true.append(data.y.item())
        all_sizes.append(data.community_size)

all_preds = np.array(all_preds)
all_true  = np.array(all_true)
all_sizes = np.array(all_sizes)

r2             = r2_score(all_true, all_preds)
mae            = mean_absolute_error(all_true, all_preds)
mse            = np.mean((all_preds - all_true) ** 2)
spearman_rho, spearman_p = spearmanr(all_true, all_preds)

# Baselines
train_mean    = np.mean([g.y.item() for g in train_graphs])
baseline1_r2  = r2_score(all_true, np.full_like(all_true, train_mean))

print(f"\n  ┌── GAT-GO Results ({len(test_graphs)} test graphs) ───────────────┐")
print(f"  │  R²         : {r2:.4f}                               │")
print(f"  │  MAE        : {mae:.4f}                               │")
print(f"  │  Spearman ρ : {spearman_rho:.4f}  (p={spearman_p:.3e})           │")
print(f"  ├── Baseline ──────────────────────────────────────────┤")
print(f"  │  Global mean R² : {baseline1_r2:.4f}                         │")
print(f"  └───────────────────────────────────────────────────────┘")


[8/8] Evaluating ...

  ┌── GAT-GO Results (240 test graphs) ───────────────┐
  │  R²         : 0.5707                               │
  │  MAE        : 0.1551                               │
  │  Spearman ρ : 0.7510  (p=8.500e-45)           │
  ├── Baseline ──────────────────────────────────────────┤
  │  Global mean R² : -0.0002                         │
  └───────────────────────────────────────────────────────┘


In [13]:
# =============================================================================
# SHAPLEY VALUES — GO LEVEL
# =============================================================================

"""
SHAPLEY FOR GAT-GO — WHAT IS DIFFERENT?
=========================================
For GAT-SS, Shapley values told us: which STRAIN is important?
For GAT-GO, we compute two levels of Shapley:

  Level 1 (STRAIN level): same as GAT-SS — which strain matters?
    The interpretation now is: which strain's GENOME drives suppression?

  Level 2 (GO FEATURE level): which GENE ORTHOLOGUES matter?
    We compute this using gradient-based attribution:
    for each GO feature, how much does a small change in that feature
    change the model's output? Large gradient = important feature.
    This is called "gradient × input" attribution.

GRADIENT-BASED GO FEATURE IMPORTANCE
--------------------------------------
For each strain in a community:
  1. Run forward pass, compute prediction
  2. Run backward pass to get gradients of prediction w.r.t. input features
  3. Importance of feature f = |gradient_f × input_f|
     (gradient tells direction of change, input tells current value)
  4. Average across all strains and communities

This is a fast approximation of Shapley values for features,
called "Integrated Gradients" or "gradient × input" attribution.
"""

print("\n  Computing Shapley values (strain level) for sample communities ...")

def compute_shapley_strain(model, data, n_permutations=N_SHAPLEY_PERMUTATIONS):
    """Compute strain-level Shapley values — same logic as GAT-SS."""
    model.eval()
    n_nodes = data.x.shape[0]
    shapley_vals = np.zeros(n_nodes)

    # Baseline: single node with mean GO feature vector
    mean_go = data.x.mean(dim=0, keepdim=True)   # [1, N_TOP_GO]
    b_edge  = torch.tensor([[0], [0]], dtype=torch.long)
    b_batch = torch.zeros(1, dtype=torch.long)
    b_data  = Data(x=mean_go, edge_index=b_edge, batch=b_batch)
    with torch.no_grad():
        baseline_pred = model(b_data).item()

    for _ in range(n_permutations):
        order     = np.random.permutation(n_nodes)
        prev_pred = baseline_pred
        coalition = []

        for node_idx in order:
            coalition.append(node_idx)
            sub_feats = data.x[coalition, :]
            n_sub = len(coalition)

            sub_src, sub_dst = [], []
            for i in range(n_sub):
                for j in range(n_sub):
                    if i != j:
                        sub_src.append(i)
                        sub_dst.append(j)

            if len(sub_src) == 0:
                sub_src = [0]; sub_dst = [0]

            sub_edge  = torch.tensor([sub_src, sub_dst], dtype=torch.long)
            sub_batch = torch.zeros(n_sub, dtype=torch.long)
            sub_data  = Data(x=sub_feats, edge_index=sub_edge, batch=sub_batch)

            with torch.no_grad():
                curr_pred = model(sub_data).item()

            shapley_vals[node_idx] += curr_pred - prev_pred
            prev_pred = curr_pred

    return shapley_vals / n_permutations


def compute_go_feature_importance(model, graph_list_sample, top_go_cols):
    """
    Compute gradient-based importance for each GO feature.

    For each community graph:
      1. Forward pass with gradients enabled
      2. Backpropagate
      3. importance = |gradient × input| for each GO feature
      4. Average across all nodes in graph and all graphs in sample

    Returns
    -------
    feature_importance : ndarray [N_TOP_GO]
        One importance score per GO feature.
        Higher = this GO gene is more important for predictions.
    """
    importance_accumulator = np.zeros(len(top_go_cols))
    n_graphs_used = 0

    for data in graph_list_sample:
        # requires_grad_(True) tells PyTorch to track gradients for this tensor.
        # Normally we don't compute gradients for inputs (only for weights).
        # Here we want to know: if I slightly change each input feature,
        # how much does the output change? That's the gradient w.r.t. inputs.
        x_inp = data.x.clone().requires_grad_(True)

        batch_tensor = torch.zeros(x_inp.shape[0], dtype=torch.long)
        d = Data(x=x_inp, edge_index=data.edge_index, batch=batch_tensor)

        model.eval()
        pred = model(d)
        # Backpropagate: compute d(pred) / d(x_inp) for all elements of x_inp
        pred.backward()

        if x_inp.grad is None:
            continue

        # x_inp.grad shape: [n_nodes, N_TOP_GO]
        # grad × input: element-wise multiplication
        # Then take absolute value and average across nodes
        grad_times_input = (x_inp.grad * x_inp).abs()   # [n_nodes, N_TOP_GO]
        node_avg = grad_times_input.mean(dim=0)          # [N_TOP_GO]
        importance_accumulator += node_avg.detach().numpy()
        n_graphs_used += 1

    if n_graphs_used > 0:
        importance_accumulator /= n_graphs_used

    return importance_accumulator


# Strain-level Shapley
sample_graphs_for_shap = []
for lo, hi in [(2, 8), (9, 20), (21, 50)]:
    bin_g = [g for g in test_graphs if lo <= g.community_size <= hi]
    n_s   = min(10, len(bin_g))
    if n_s > 0:
        sample_graphs_for_shap.extend(random.sample(bin_g, n_s))

shapley_results = []
for data in sample_graphs_for_shap:
    shap_vals = compute_shapley_strain(model, data)
    for i, (sid, sv) in enumerate(zip(data.strain_ids, shap_vals)):
        shapley_results.append({
            "community_id"  : data.community_id,
            "community_size": data.community_size,
            "strain_id"     : sid,
            "shapley_value" : sv,
            "abs_shapley"   : abs(sv),
        })

df_shap = pd.DataFrame(shapley_results)

# GO feature importance (gradient-based)
print("  Computing GO feature importance (gradient × input) ...")
go_importance = compute_go_feature_importance(
    model, random.sample(test_graphs, min(50, len(test_graphs))), top_go_cols
)

# Top GO features by importance
top_go_importance_idx = np.argsort(go_importance)[::-1][:20]
top_go_names = [top_go_cols[i] for i in top_go_importance_idx]
top_go_scores = go_importance[top_go_importance_idx]

print(f"  Top 5 most important GO features:")
for name, score in zip(top_go_names[:5], top_go_scores[:5]):
    print(f"    {name}: {score:.5f}")


  Computing Shapley values (strain level) for sample communities ...
  Computing GO feature importance (gradient × input) ...
  Top 5 most important GO features:
    GO_00038: 0.00574
    GO_00138: 0.00490
    GO_00298: 0.00472
    GO_00422: 0.00461
    GO_00475: 0.00437


In [14]:
# =============================================================================
# FIGURES
# =============================================================================
 
print("\n  Generating figures ...")
 
# ── Figure 1: Training curves ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("GAT-GO v6 Training", fontsize=13, fontweight="bold")
 
epochs_x = range(1, len(train_losses) + 1)
axes[0].plot(epochs_x, train_losses, color=TEAL,   lw=2, label="Train")
axes[0].plot(epochs_x, test_losses,  color=ORANGE, lw=2, label="Test")
axes[0].set_title("Loss Curves (Huber Loss, δ=0.15)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Huber Loss")
axes[0].set_yscale("log")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
 
axes[1].plot(list(epochs_x)[-100:], train_losses[-100:], color=TEAL,   lw=2, label="Train")
axes[1].plot(list(epochs_x)[-100:], test_losses[-100:],  color=ORANGE, lw=2, label="Test")
axes[1].set_title("Final 100 Epochs (zoomed)")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
 
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig1_training_curves_go_v6.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("  → fig1_training_curves_go_v6.png")
 
# ── Figure 2: Prediction performance ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("GAT-GO v6 Prediction Performance", fontsize=13, fontweight="bold")
 
sc = axes[0].scatter(all_true, all_preds, c=all_sizes,
                     cmap="viridis_r", s=40, alpha=0.6, edgecolors="white")
plt.colorbar(sc, ax=axes[0], label="Community size")
axes[0].plot([0, 1], [0, 1], "k--", lw=1.5)
axes[0].set_title(f"Predicted vs Actual\nR²={r2:.3f}  MAE={mae:.3f}  ρ={spearman_rho:.3f}")
axes[0].set_xlabel("Actual suppression")
axes[0].set_ylabel("Predicted suppression")
axes[0].grid(True, alpha=0.3)
 
# <<< CHANGE: size bins updated to match v6 community sizes >>>
# Original bins [1,4,8,15,24,35,50] were for the original dataset
# with sizes up to 50. v6 has sizes {2,4,6,8,12,16,24,48}.
# New bin edges align with the actual v6 size distribution.
bins   = [1, 4, 8, 12, 16, 24, 48]
labels = ["2–4", "5–8", "9–12", "13–16", "17–24", "25–48"]
bin_r2 = []
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (all_sizes > lo) & (all_sizes <= hi)
    if mask.sum() >= 3:
        bin_r2.append(r2_score(all_true[mask], all_preds[mask]))
    else:
        bin_r2.append(np.nan)
 
axes[1].bar(range(len(labels)), bin_r2,
            color=[TEAL if not np.isnan(v) else GREY for v in bin_r2],
            edgecolor="white", width=0.6, alpha=0.85)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels(labels)
axes[1].set_title("R² per Community Size Bin")
axes[1].set_xlabel("Community size bin")
axes[1].set_ylabel("R²")
axes[1].grid(True, axis="y", alpha=0.3)
 
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig2_prediction_performance_go_v6.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("  → fig2_prediction_performance_go_v6.png")
 
# ── Figure 3: GO Feature importance ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("GAT-GO v6 Gene Orthologue Feature Importance\n"
             "(gradient × input attribution — which genes drive predictions?)",
             fontsize=13, fontweight="bold")
 
axes[0].barh(range(20), top_go_scores[::-1],
             color=TEAL, alpha=0.8, edgecolor="white")
axes[0].set_yticks(range(20))
axes[0].set_yticklabels(top_go_names[::-1], fontsize=8)
axes[0].set_title("A) Top 20 Most Important GO Features\n"
                  "(gradient × input importance score)")
axes[0].set_xlabel("Importance score")
axes[0].grid(True, axis="x", alpha=0.3)
 
axes[1].hist(go_importance, bins=40, color=TEAL, alpha=0.75, edgecolor="white")
axes[1].axvline(go_importance[top_go_importance_idx[-1]], color=ORANGE,
                lw=2, ls="--", label=f"Top {N_TOP_GO} cutoff")
axes[1].set_title("B) Distribution of All GO Feature Importances\n"
                  "(most features have near-zero importance)")
axes[1].set_xlabel("Importance score")
axes[1].set_ylabel("Number of GO features")
axes[1].legend()
axes[1].grid(True, axis="y", alpha=0.3)
 
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig3_go_feature_importance_go_v6.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("  → fig3_go_feature_importance_go_v6.png")
 
# ── Figure 4: Shapley analysis ────────────────────────────────────────────
if len(df_shap) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("GAT-GO v6 Shapley Values (strain-level genomic contribution)",
                 fontsize=13, fontweight="bold")
 
    # <<< CHANGE: Shapley size bins updated upper bound 50 → 48 >>>
    bins_shap   = [1, 8, 20, 48]
    labels_shap = ["2–8", "9–20", "21–48"]
 
    bin_shap = df_shap.groupby(
        pd.cut(df_shap["community_size"], bins=bins_shap, labels=labels_shap)
    )["abs_shapley"].agg(["mean", "std"]).reset_index()
 
    axes[0].plot(range(len(bin_shap)), bin_shap["mean"],
                 color=TEAL, lw=2.5, marker="o", markersize=8,
                 markerfacecolor="white", markeredgewidth=2, markeredgecolor=TEAL)
    axes[0].fill_between(range(len(bin_shap)),
                         bin_shap["mean"] - bin_shap["std"],
                         bin_shap["mean"] + bin_shap["std"],
                         alpha=0.15, color=TEAL)
    axes[0].set_xticks(range(len(bin_shap)))
    axes[0].set_xticklabels(labels_shap, fontsize=9)
    axes[0].set_title("A) Mean |Shapley| vs Community Size\n"
                      "(should decrease = genomic roles stabilise with diversity)")
    axes[0].set_xlabel("Community size bin")
    axes[0].set_ylabel("Mean absolute Shapley value")
    axes[0].grid(True, alpha=0.3)
 
    sns.violinplot(
        data=df_shap,
        x=pd.cut(df_shap["community_size"], bins=bins_shap, labels=labels_shap),
        y="shapley_value",
        color=TEAL, alpha=0.6, linewidth=1.2,
        inner="quartile", ax=axes[1]
    )
    axes[1].axhline(0, color=GREY, lw=1.2, ls="--")
    axes[1].set_title("B) Shapley Distribution per Size Bin\n"
                      "(genomic contributions stabilise in larger communities)")
    axes[1].set_xlabel("Community size bin")
    axes[1].set_ylabel("Shapley value")
    axes[1].tick_params(axis="x", labelsize=8)
    axes[1].grid(True, axis="y", alpha=0.3)
 
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "fig4_shapley_analysis_go_v6.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  → fig4_shapley_analysis_go_v6.png")
 
# ── Figure 5: Node embeddings + GO correlation heatmap ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("GAT-GO v6 Node Embeddings\n"
             "(what the model learned about each strain's genomic role)",
             fontsize=13, fontweight="bold")
 
largest = max(test_graphs, key=lambda g: g.community_size)
model.eval()
with torch.no_grad():
    emb = model.get_node_embeddings(largest).numpy()   # [N_nodes, 32]
 
pca    = PCA(n_components=2)
coords = pca.fit_transform(emb)
 
# Colour by gene richness: fraction of GO features that are non-zero.
# (largest.x > 0) gives a boolean tensor; .float() converts True/False → 1/0.
# .mean(dim=1) averages across the GO feature dimension → one value per node.
gene_richness = (largest.x > 0).float().mean(dim=1).numpy()
sc = axes[0].scatter(coords[:, 0], coords[:, 1], c=gene_richness,
                     cmap="YlOrRd", s=80, edgecolors="white",
                     linewidths=0.5, alpha=0.85)
plt.colorbar(sc, ax=axes[0], label="Gene richness (fraction of GO features present)")
axes[0].set_title(f"A) Node Embeddings (PCA)\nN={largest.community_size} community\n"
                  f"Colour = gene richness")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[0].grid(True, alpha=0.3)
 
# Correlation heatmap of top 15 GO features across 100 monoculture strains.
# np.corrcoef(X.T): computes pairwise Pearson correlations between columns of X.
# X.T has shape [N_features, N_strains] so corrcoef gives [N_features, N_features].
top15_cols   = top_go_cols[:15]
go_corr_data = df_mono[top15_cols].values
go_corr      = np.corrcoef(go_corr_data.T)
 
sns.heatmap(go_corr, ax=axes[1], cmap="RdBu_r",
            center=0, vmin=-1, vmax=1,
            xticklabels=[c.replace("GO_", "") for c in top15_cols],
            yticklabels=[c.replace("GO_", "") for c in top15_cols],
            linewidths=0.3, cbar_kws={"label": "Pearson correlation"})
axes[1].set_title("B) Correlation Between Top 15 GO Features\n"
                  "(correlated genes = tend to appear together in strains)")
axes[1].tick_params(axis="both", labelsize=7)
 
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig5_embeddings_go_corr_go_v6.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("  → fig5_embeddings_go_corr_go_v6.png")
 
 


  Generating figures ...
  → fig1_training_curves_go_v6.png
  → fig2_prediction_performance_go_v6.png
  → fig3_go_feature_importance_go_v6.png
  → fig4_shapley_analysis_go_v6.png
  → fig5_embeddings_go_corr_go_v6.png


In [16]:
# =============================================================================
# SAVE MODEL
# =============================================================================

import pickle
model_path = os.path.join(OUTPUT_DIR, "gat_go_v6_model.pt")
torch.save(model.state_dict(), model_path)
 
# Save the scaler and selected feature names.
# These are required to preprocess new data when you reload the model.
# Without them you can't reproduce the exact same normalisation.
scaler_path  = os.path.join(OUTPUT_DIR, "go_scaler_v6.pkl")
feature_path = os.path.join(OUTPUT_DIR, "top_go_cols_v6.pkl")
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
with open(feature_path, "wb") as f:
    pickle.dump(top_go_cols, f)
 
df_shap.to_csv(os.path.join(OUTPUT_DIR, "shapley_values_gat_go_v6.csv"), index=False)
 
print(f"\n  Model saved     : {model_path}")
print(f"  Scaler saved    : {scaler_path}")
print(f"  GO cols saved   : {feature_path}")
 
print("\n" + "=" * 65)
print("  GAT-GO v6 COMPLETE")
print("=" * 65)
print(f"  Final R²        : {r2:.4f}")
print(f"  Final MAE       : {mae:.4f}")
print(f"  Spearman ρ      : {spearman_rho:.4f}")
print(f"  Beats baseline  : {'YES' if r2 > baseline1_r2 else 'NO'}")
print(f"  All outputs in  : {OUTPUT_DIR}")
print("=" * 65)


  Model saved     : C:\Users\Pratham Shah\Downloads\Document dataset\Final_MyData\GAT_GO_Results_1\gat_go_v6_model.pt
  Scaler saved    : C:\Users\Pratham Shah\Downloads\Document dataset\Final_MyData\GAT_GO_Results_1\go_scaler_v6.pkl
  GO cols saved   : C:\Users\Pratham Shah\Downloads\Document dataset\Final_MyData\GAT_GO_Results_1\top_go_cols_v6.pkl

  GAT-GO v6 COMPLETE
  Final R²        : 0.5707
  Final MAE       : 0.1551
  Spearman ρ      : 0.7510
  Beats baseline  : YES
  All outputs in  : C:\Users\Pratham Shah\Downloads\Document dataset\Final_MyData\GAT_GO_Results_1
